# 实验 2 — 改列类型会怎样？

**目标**：理解 dbt view vs table 在列类型变更时的行为差异。

**操作步骤**：
1. 先跑下面的 cell 看看当前 `stg_ecb_rates` 的 schema
2. 把 `dbt_project/models/staging/stg_ecb_rates.sql` 里的 `rate::double` 改成 `rate::varchar`
3. 再跑 `dbt build`，观察结果
4. 改回来

In [ ]:
import duckdb
import subprocess

DB = "../data/sandbox.duckdb"
con = duckdb.connect(DB)

print("当前 stg_ecb_rates schema（view，实时反映 SQL）：")
con.execute("DESCRIBE main.stg_ecb_rates").df()

In [ ]:
# 在 stg_ecb_rates.sql 里临时把 rate::double 改成 rate::varchar
# 然后运行这个 cell

result = subprocess.run(
    ["uv", "run", "dbt", "build"],
    capture_output=True, text=True, cwd="../dbt_project",
    env={**__import__('os').environ, "DBT_PROFILES_DIR": "."}
)
print(result.stdout[-3000:])  # 只打印末尾部分
print(result.stderr[-1000:] if result.returncode != 0 else "")

In [ ]:
# 看 fct_daily_rates 的 not_null test 对 rate 字段会发生什么？
# （varchar 类型的 NULL 检查和 double 有什么区别？）

# 改回去之后再检查 schema
con2 = duckdb.connect(DB)
print("修改后 stg_ecb_rates schema：")
con2.execute("DESCRIBE main.stg_ecb_rates").df()

## 思考

- `stg_ecb_rates` 是 **view**，列类型改变立即生效（下次查询时重新计算）
- `fct_daily_rates` 是 **table**，需要 `dbt run` 才能更新物化结果
- 如果 `fct_daily_rates` 里的 `rate` 变成了 varchar，`rate_change_pct`（数学计算）会报什么错？